# T10 — DeBERTa-v3-base on Colab

`deberta-v3-base` cannot train on the local RTX 3050: in fp32 it needs 2.94 GB for
weights + grads + two AdamW moments alone, on 3.65 GB usable, and every batch-size
and gradient-checkpointing combination OOMs. A T4 (15 GB) has room.

**Before running this, push the local repo.** This notebook clones from GitHub and
needs two commits that may still be local:

- `179a678` — `--model` override (without it there is no way to select the encoder)
- `57f1b78` — forces fp32 (**without it DeBERTa produces NaN on the first optimizer
  step**, because its checkpoint ships fp16 and AdamW's `denom` underflows to zero)

Data is cloned from the public ACTER repo at the same commit this project uses
(`f05b09e`, version 1.5). Nothing non-commercial is redistributed and nothing is uploaded.

**Runtime → Change runtime type → T4 GPU** before you start.


In [ ]:
# 1. GPU check -- stop here if this is a CPU runtime
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'


In [ ]:
# 2. Clone the project and the corpus (ACTER pinned to the same commit as local)
%cd /content
!rm -rf ate-acter
!git clone -q https://github.com/ahmedwaleedaref/ATE-ACTER.git ate-acter
%cd /content/ate-acter
!git log --oneline -1

!mkdir -p data/raw
!git clone -q https://github.com/AylaRT/ACTER.git data/raw/ACTER
!cd data/raw/ACTER && git checkout -q f05b09e985cad37eeaa8daa8b3f383197aa5324e && git log --oneline -1

# the two commits this notebook depends on
import subprocess
log = subprocess.run(['git','log','--oneline','-40'], capture_output=True, text=True).stdout
assert '--model override' in log, 'repo predates --model: push 179a678 first'
assert 'forces fp32' in log, 'repo predates the fp32 fix: push 57f1b78 first, or DeBERTa will NaN'
print('\nboth required commits present')


In [ ]:
# 3. Pinned installs. Same direct versions as the local .venv, so T10 stays
# comparable across machines. Colab's Python differs, so transitive pins from
# requirements.txt (built for cp314) are not reusable.
!pip install -q \
  torch==2.14.0 \
  transformers==5.16.1 \
  tokenizers==0.23.1 \
  safetensors==0.8.0 \
  huggingface_hub==1.29.0 \
  seqeval==1.2.2 \
  scikit-learn==1.9.0 \
  numpy==2.5.2 \
  PyYAML==6.0.3 \
  sentencepiece==0.2.2 \
  protobuf==7.36.0
print('installed')


In [ ]:
# 4. Verify the environment matches local, and that the harness still passes.
# Any mismatch here is a confound for T10 and must be recorded, not ignored.
import torch, transformers, sys
LOCAL = {'torch': '2.14.0', 'transformers': '5.16.1'}
here = {'torch': torch.__version__.split('+')[0], 'transformers': transformers.__version__}
for k, v in LOCAL.items():
    mark = 'match' if here[k] == v else f'DIFFERS from local {v} -- record this in E04'
    print(f'{k:14} {here[k]:12} {mark}')
print('python', sys.version.split()[0], '(local was 3.14.4)')
!python -m pytest tests/ -q 2>&1 | tail -3


In [ ]:
# 5. Single-seed probe. Confirms DeBERTa trains at all before committing to five.
# Watch epoch 1: a finite train_loss means the fp32 fix is doing its job.
!python -m src.models.run_train --seed 42 \
    --model microsoft/deberta-v3-base \
    --group t10/deberta-v3-base \
    --reason 'T10 deberta-v3-base, T9 config, Colab T4'


In [ ]:
# 6. Remaining four seeds. Only run this once the probe above looks sane.
for s in (43, 44, 45, 46):
    print(f'===== seed {s} =====')
    !python -m src.models.run_train --seed {s} \
        --model microsoft/deberta-v3-base \
        --group t10/deberta-v3-base \
        --reason 'T10 deberta-v3-base, T9 config, Colab T4'


In [ ]:
# 7. Cell stats, then download. Commit the JSONs into results/runs/t10/ locally.
!python -m src.aggregate --cell results/runs/t10/deberta-v3-base

!zip -qr /content/t10_deberta.zip results/runs/t10 results/test_evaluations.log
from google.colab import files
files.download('/content/t10_deberta.zip')
